In [67]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

### I. 40 Point Games - number by season

In [68]:
forty_burgers = pd.read_csv('40_point_games.csv')

forty_burgers

,Rk,Player,PTS,Date,Age,Team,Unnamed: 6,Opp,Result,GS,...,STL,BLK,TOV,PF,PTS.1,GmSc,BPM,+/-,Pos.,Player-additional
0,1,Kobe Bryant,81,2006-01-22,27-152,LAL,NaN,TOR,W 122-104,*,...,3.0,1.0,3.0,1.0,81,63.5,33.6,25.0,G-F,bryanko01
1,2,Luka Dončić,73,2024-01-26,24-332,DAL,@,ATL,W 148-143,*,...,1.0,0.0,4.0,1.0,73,64.0,20.7,13.0,G-F,doncilu01
2,3,Damian Lillard,71,2023-02-26,32-226,POR,NaN,HOU,W 131-114,*,...,0.0,0.0,2.0,0.0,71,57.6,23.5,21.0,G,lillada01
3,4,Donovan Mitchell,71,2023-01-02,26-117,CLE,NaN,CHI,W 145-134 (OT),*,...,0.0,1.0,4.0,3.0,71,60.8,23.0,19.0,G,mitchdo01
4,5,David Robinson,71,1994-04-24,28-261,SAS,@,LAC,W 112-97,*,...,0.0,2.0,8.0,2.0,71,51.8,19.7,NaN,C,robinda01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3833,3834,Giannis Antetokounmpo,41,2025-11-07,30-336,MIL,NaN,CHI,W 126-110,*,...,2.0,2.0,4.0,1.0,41,35.2,14.7,16.0,F-G,antetgi01
3834,3835,Trey Murphy III,41,2025-11-08,25-143,NOP,@,SAS,L 119-126,*,...,1.0,0.0,1.0,1.0,41,37.2,15.4,-14.0,F,murphtr02
3835,1,Cade Cunningham,46,2025-11-10,24-046,DET,NaN,WAS,W 137-135 (OT),*,...,5.0,2.0,2.0,4.0,46,35.4,8.7,10.0,G,cunnica01
3836,2,Grayson Allen,42,2025-11-10,30-033,PHO,NaN,NOP,W 121-98,*,...,3.0,0.0,1.0,0.0,42,39.9,36.7,22.0,G,allengr01


In [69]:
forty_burgers.rename(columns={'Unnamed: 6': 'Road'}, inplace=True)

forty_burgers.dtypes

Rk                     int64
Player                object
PTS                    int64
Date                  object
Age                   object
Team                  object
Road                  object
Opp                   object
Result                object
GS                    object
MP                   float64
FG                     int64
FGA                  float64
FG%                  float64
2P                     int64
2PA                  float64
2P%                  float64
3P                     int64
3PA                  float64
3P%                  float64
FT                     int64
FTA                    int64
FT%                  float64
TS%                  float64
ORB                  float64
DRB                  float64
TRB                  float64
AST                  float64
STL                  float64
BLK                  float64
TOV                  float64
PF                   float64
PTS.1                  int64
GmSc                 float64
BPM           

In [70]:
forty_burgers.columns

Index(['Rk', 'Player', 'PTS', 'Date', 'Age', 'Team', 'Road', 'Opp', 'Result',
       'GS', 'MP', 'FG', 'FGA', 'FG%', '2P', '2PA', '2P%', '3P', '3PA', '3P%',
       'FT', 'FTA', 'FT%', 'TS%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK',
       'TOV', 'PF', 'PTS.1', 'GmSc', 'BPM', '+/-', 'Pos.',
       'Player-additional'],
      dtype='object')

In [71]:
numeric_cols = ['PTS', 'MP', 'FG', 'FGA', 'FG%', '2P', '2PA', '2P%', '3P', '3PA', '3P%',
       'FT', 'FTA', 'FT%', 'TS%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK',
       'TOV', 'PF', 'PTS.1', 'GmSc', 'BPM', '+/-']

for col in numeric_cols:
    forty_burgers[col] = pd.to_numeric(forty_burgers[col], errors='coerce')
    
forty_burgers.dtypes

Rk                     int64
Player                object
PTS                    int64
Date                  object
Age                   object
Team                  object
Road                  object
Opp                   object
Result                object
GS                    object
MP                   float64
FG                     int64
FGA                  float64
FG%                  float64
2P                     int64
2PA                  float64
2P%                  float64
3P                     int64
3PA                  float64
3P%                  float64
FT                     int64
FTA                    int64
FT%                  float64
TS%                  float64
ORB                  float64
DRB                  float64
TRB                  float64
AST                  float64
STL                  float64
BLK                  float64
TOV                  float64
PF                   float64
PTS.1                  int64
GmSc                 float64
BPM           

In [72]:
# turn `Road` column into boolean that's True if the value is '@' (indicating an away game)
forty_burgers['Road'] = forty_burgers['Road'] == '@'

# turn `Date` column from string like "2006-01-22" into datetime
forty_burgers['Date'] = pd.to_datetime(forty_burgers['Date'])

forty_burgers.dtypes

Rk                            int64
Player                       object
PTS                           int64
Date                 datetime64[ns]
Age                          object
Team                         object
Road                           bool
Opp                          object
Result                       object
GS                           object
MP                          float64
FG                            int64
FGA                         float64
FG%                         float64
2P                            int64
2PA                         float64
2P%                         float64
3P                            int64
3PA                         float64
3P%                         float64
FT                            int64
FTA                           int64
FT%                         float64
TS%                         float64
ORB                         float64
DRB                         float64
TRB                         float64
AST                         

### II. 40 point games - scoring by type

In [73]:
forty_burgers['pts_from_3'] = forty_burgers['3P'] * 3
forty_burgers['pts_from_2'] = forty_burgers['2P'] * 2

In [74]:
# compute season as the season-ending year (NBA seasons start in Oct)
forty_burgers['season'] = forty_burgers['Date'].dt.year + (forty_burgers['Date'].dt.month >= 10).astype(int)

# points from free throws (FT column is made free throws)
forty_burgers['pts_from_ft'] = forty_burgers['FT']

# aggregate by season and compute percentages
season_points = forty_burgers.groupby('season').agg(
    total_pts=('PTS', 'sum'),
    pts_3=('pts_from_3', 'sum'),
    pts_2=('pts_from_2', 'sum'),
    pts_ft=('pts_from_ft', 'sum'),
)

season_points[['pct_3', 'pct_2', 'pct_ft']] = (
    season_points[['pts_3', 'pts_2', 'pts_ft']].div(season_points['total_pts'], axis=0) * 100
).round(2)

# show results
season_points = season_points.sort_index()
season_points[['pct_3', 'pct_2', 'pct_ft']]

,pct_3,pct_2,pct_ft
season,,,
1980,3.38,77.14,19.48
1981,1.44,75.84,22.72
1982,1.38,76.55,22.06
1983,2.87,75.59,21.54
1984,2.50,75.45,22.05
1985,3.47,73.85,22.68
1986,3.42,73.20,23.38
1987,4.09,70.66,25.25
1988,4.81,72.78,22.41


In [75]:
# cleanup for export

# drop unneeded columns and rename for clarity
season_points.drop(columns=['total_pts', 'pts_3', 'pts_2', 'pts_ft'], inplace=True)

# rename columns
season_points.rename(columns={'pct_3':'Three Pointers', 'pct_2':'Two Pointers', 'pct_ft':'Free Throws'}, inplace=True)

# make 'season' a column instead of index
season_points['Season'] = season_points.index
season_points.reset_index(drop=True, inplace=True)

# subtract a year from season
season_points['Season'] = season_points['Season'] - 1

season_points

,Three Pointers,Two Pointers,Free Throws,Season
0,3.38,77.14,19.48,1979
1,1.44,75.84,22.72,1980
2,1.38,76.55,22.06,1981
3,2.87,75.59,21.54,1982
4,2.50,75.45,22.05,1983
5,3.47,73.85,22.68,1984
6,3.42,73.20,23.38,1985
7,4.09,70.66,25.25,1986
8,4.81,72.78,22.41,1987
9,5.55,70.42,24.03,1988


In [76]:
season_points.to_csv('season_points_summary.csv')

### Running Totals

In [77]:
import numpy as np

# Create a day-of-season for each game (days since Oct 1 of the season's start year)
# Season 2024 starts in Oct 2023, so start_year = season - 1
forty_burgers['season_start_year'] = forty_burgers['season'] - 1
forty_burgers['season_start_date'] = pd.to_datetime(
    forty_burgers['season_start_year'].astype(str) + '-10-01'
)
forty_burgers['days_into_season'] = (forty_burgers['Date'] - forty_burgers['season_start_date']).dt.days

# Create date range: Oct 1 to Dec 1 (62 days, day 0 to day 61)
date_range = pd.date_range('2000-10-01', '2000-12-01', freq='D')
days_into_season_range = np.arange(0, 62)  # 0 to 61 days

# Get unique seasons
seasons = sorted(forty_burgers['season'].unique())

# Create the running totals dataframe
running_totals = pd.DataFrame(index=date_range.strftime('%b %d'))

for season in seasons:
    # Get games for this season
    season_games = forty_burgers[forty_burgers['season'] == season]
    
    # For each day in our range, count games up to that day
    cumulative_counts = []
    for day in days_into_season_range:
        count = (season_games['days_into_season'] <= day).sum()
        cumulative_counts.append(count)
    
    running_totals[season] = cumulative_counts
    
# rename numeric season columns like 1980 -> "1979 - 1980"
mapping = {}
for c in running_totals.columns:
    try:
        year = int(c)
    except Exception:
        # skip non-integer column names
        continue
    mapping[c] = f"{year - 1} - {year}"

running_totals.rename(columns=mapping, inplace=True)

# add the day labels as a column
running_totals.insert(0, 'Date', running_totals.index)
# make it datetime format
running_totals['Date'] = pd.to_datetime(running_totals['Date'], format='%b %d')

# erase all rows where every column (except Date) is zero
running_totals = running_totals[(running_totals.drop(columns=['Date']) != 0).any(axis=1)]

# in the Date column, set the year to 2000 for consistency
running_totals['Date'] = running_totals['Date'].apply(lambda d: d.replace(year=2000))

# in the 2025-2026 column, set any value where the date is after today to NaN
from datetime import datetime
today = datetime.today()
if '2025 - 2026' in running_totals.columns:
    running_totals.loc[running_totals['Date'] > today.replace(year=2000), '2025 - 2026'] = np.nan
    
# make all cols except Date float
for col in running_totals.columns:
    if col != 'Date':
        running_totals[col] = running_totals[col].astype(float)

# Display
running_totals

,Date,1979 - 1980,1980 - 1981,1981 - 1982,1982 - 1983,1983 - 1984,1984 - 1985,1985 - 1986,1986 - 1987,1987 - 1988,...,2016 - 2017,2017 - 2018,2018 - 2019,2019 - 2020,2020 - 2021,2021 - 2022,2022 - 2023,2023 - 2024,2024 - 2025,2025 - 2026
Oct 04,2000-10-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Oct 05,2000-10-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Oct 06,2000-10-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Oct 07,2000-10-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Oct 08,2000-10-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Oct 09,2000-10-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Oct 10,2000-10-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Oct 11,2000-10-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Oct 12,2000-10-12,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
Oct 13,2000-10-13,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0


In [78]:
# export to CSV
running_totals.to_csv('running_totals_40_point_games.csv', index=False)

### Players per season

In [79]:
forty_burgers

,Rk,Player,PTS,Date,Age,Team,Road,Opp,Result,GS,...,+/-,Pos.,Player-additional,pts_from_3,pts_from_2,season,pts_from_ft,season_start_year,season_start_date,days_into_season
0,1,Kobe Bryant,81,2006-01-22,27-152,LAL,False,TOR,W 122-104,*,...,25.0,G-F,bryanko01,21,42,2006,18,2005,2005-10-01,113
1,2,Luka Dončić,73,2024-01-26,24-332,DAL,True,ATL,W 148-143,*,...,13.0,G-F,doncilu01,24,34,2024,15,2023,2023-10-01,117
2,3,Damian Lillard,71,2023-02-26,32-226,POR,False,HOU,W 131-114,*,...,21.0,G,lillada01,39,18,2023,14,2022,2022-10-01,148
3,4,Donovan Mitchell,71,2023-01-02,26-117,CLE,False,CHI,W 145-134 (OT),*,...,19.0,G,mitchdo01,21,30,2023,20,2022,2022-10-01,93
4,5,David Robinson,71,1994-04-24,28-261,SAS,True,LAC,W 112-97,*,...,NaN,C,robinda01,3,50,1994,18,1993,1993-10-01,205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3833,3834,Giannis Antetokounmpo,41,2025-11-07,30-336,MIL,False,CHI,W 126-110,*,...,16.0,F-G,antetgi01,3,30,2026,8,2025,2025-10-01,37
3834,3835,Trey Murphy III,41,2025-11-08,25-143,NOP,True,SAS,L 119-126,*,...,-14.0,F,murphtr02,15,20,2026,6,2025,2025-10-01,38
3835,1,Cade Cunningham,46,2025-11-10,24-046,DET,False,WAS,W 137-135 (OT),*,...,10.0,G,cunnica01,6,24,2026,16,2025,2025-10-01,40
3836,2,Grayson Allen,42,2025-11-10,30-033,PHO,False,NOP,W 121-98,*,...,22.0,G,allengr01,30,4,2026,8,2025,2025-10-01,40


In [80]:
# calculate the number of 40-point games per player per season
player_season_counts = forty_burgers.groupby(['Player', 'season']).size().reset_index(name='40_point_games')
player_season_counts

# pivot the table so that each player is a column and each season is a row
player_season_pivot = player_season_counts.pivot(index='season', columns='Player', values='40_point_games').fillna(0).astype(int)
player_season_pivot

Player,Aaron Brooks,Aaron Gordon,Aaron Wiggins,Acie Earl,Adrian Dantley,Al Harrington,Al Jefferson,Alex English,Allan Houston,Allen Crabbe,...,Walt Williams,Walter Davis,Wayman Tisdale,Willie Burton,World B. Free,Xavier McDaniel,Yao Ming,Zach LaVine,Zach Randolph,Zion Williamson
season,,,,,,,,,,,,,,,,,,,,,
1980,0,0,0,0,4,0,0,1,0,0,...,0,1,0,0,8,0,0,0,0,0
1981,0,0,0,0,10,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1982,0,0,0,0,13,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1983,0,0,0,0,3,0,0,6,0,0,...,0,0,0,0,0,0,0,0,0,0
1984,0,0,0,0,11,0,0,3,0,0,...,0,3,0,0,1,0,0,0,0,0
1985,0,0,0,0,4,0,0,7,0,0,...,0,0,0,0,1,0,0,0,0,0
1986,0,0,0,0,5,0,0,9,0,0,...,0,1,0,0,1,0,0,0,0,0
1987,0,0,0,0,1,0,0,6,0,0,...,0,1,0,0,0,2,0,0,0,0
1988,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [81]:
player_season_pivot.to_csv('players_season_pivot.csv')

In [82]:
# calculate number of players with at least one 40-point game per season
players_per_season = player_season_counts.groupby('season').size().reset_index(name='num_players')
players_per_season

,season,num_players
0,1980,26
1,1981,28
2,1982,19
3,1983,23
4,1984,26
5,1985,29
6,1986,24
7,1987,25
8,1988,24
9,1989,24


In [83]:
players_per_season['season'] = players_per_season['season'] - 1

In [84]:
players_per_season.to_csv('players_per_season.csv', index=False)

### Stat: players to hit 10 threes

In [87]:
threes = pd.read_csv('10_threes.csv')

threes

,Rk,Player,3P,Date,Age,Team,Unnamed: 6,Opp,Result,GS,...,STL,BLK,TOV,PF,PTS,GmSc,BPM,+/-,Pos.,Player-additional
0,1,Klay Thompson,14,2018-10-29,28-263,GSW,@,CHI,W 149-124,*,...,2,0,2,2,52,39.3,24.5,28.0,G-F,thompkl01
1,2,Stephen Curry,13,2016-11-07,28-238,GSW,NaN,NOP,W 116-106,*,...,2,0,4,2,46,37.2,24.6,7.0,G,curryst01
2,3,Zach LaVine,13,2019-11-23,24-258,CHI,@,CHO,W 116-115,*,...,2,0,4,5,49,33.4,19.5,15.0,G-F,lavinza01
3,4,Damian Lillard,13,2023-02-26,32-226,POR,NaN,HOU,W 131-114,*,...,0,0,2,0,71,57.6,23.5,21.0,G,lillada01
4,5,Kobe Bryant,12,2003-01-07,24-137,LAL,NaN,SEA,W 119-98,*,...,1,1,2,2,45,33.7,19.2,19.0,G-F,bryanko01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,104,Klay Thompson,10,2015-12-08,25-303,GSW,@,IND,W 131-123,*,...,0,1,4,2,39,32.1,18.3,19.0,G-F,thompkl01
104,105,Klay Thompson,10,2016-03-18,26-039,GSW,@,DAL,W 130-112,*,...,2,1,3,3,39,30.9,17.3,2.0,G-F,thompkl01
105,106,Karl-Anthony Towns,10,2024-01-22,28-068,MIN,NaN,CHO,L 125-128,*,...,0,0,7,2,62,41.5,9.8,0.0,C-F,townska01
106,107,Fred VanVleet,10,2024-03-23,30-027,HOU,NaN,UTA,W 147-119,*,...,2,0,2,3,34,31.8,25.4,29.0,G,vanvlfr01


In [88]:
threes['Player'].nunique()

58

### Number of 40 point games in 2024-25|

In [85]:
forty_point_games_2425 = forty_burgers[forty_burgers['season_start_year'] == 2024]

# create a game identifier column by combining `Date` and the combination of `Team` and `Opp`. If team A is `Team ` and team B is `Opp`, it should mean the exact same game as when team B is `Team` and team A is `Opp`.
forty_point_games_2425['game_id'] = forty_point_games_2425.apply(
    lambda row: f"{row['Date'].strftime('%Y-%m-%d')}_{min(row['Team'], row['Opp'])}_{
        max(row['Team'], row['Opp'])}", axis=1
)

/var/folders/jd/5wy1jytx2pg8j4jr12tl0k_m0000gq/T/ipykernel_17035/3529907113.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forty_point_games_2425['game_id'] = forty_point_games_2425.apply(


In [86]:
forty_point_games_2425['game_id'].nunique()

144